<a href="https://colab.research.google.com/github/aodm26/gpt-oss/blob/main/GPT_OSS_Sentiment_analysis_2_min_RSS_gui_working.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT-OSS-20B Sentiment Analysis

**Dataset:** `test_split 1.csv` — 484 financial headlines (pre-labelled)  
**Labels:** `positive` / `negative` / `neutral` (text, lowercase)  
**Label split:** neutral=134 · positive=134 · negative=134

### Speed decisions
| Setting | Value | Reason |
|---|---|---|
| `load_in_4bit` | `True` | ~4× less VRAM for weights |
| `max_seq_length` | `1024` | Halves KV-cache pre-allocation vs 2048 |
| `max_new_tokens` | `96` | Label word + 1-sentence reason fits easily |
| `do_sample` | `False` | Greedy decode — no sampling overhead |
| Batch size | `8` | Batching causes KV-cache OOM on T4 |
| Prompt | Forces single label word on last line | Zero-ambiguity parsing |


## 1 · Install Dependencies

In [ ]:

import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo


## 2 · Load Model (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

# --- OPTIMIZATIONS FOR A100/L4 ---
# 1. Critical Lever: Set padding to LEFT for batching to work correctly
# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/gpt-oss-20b-unsloth-bnb-4bit",  # 4-bit — fits T4
    dtype          = torch.bfloat16,
    max_seq_length = 1024,   # small = less KV-cache VRAM
    load_in_4bit   = True,
    full_finetuning= False,
)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# 2. Precision Lever: Use BFloat16 for A100 (more stable than FP16)
# This is now handled directly in from_pretrained above.

# 3. Unsloth Inference Optimization
FastLanguageModel.for_inference(model)
print("✓ Optimizations applied: BF16 + Left-Padding.")
print("✓ Model ready.")

## 3 · Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/test_split 1.csv')

# Normalise labels to Title case so they match parse_label() output
df['sentiment'] = df['sentiment'].str.strip().str.capitalize()

headlines_list  = df['headline'].tolist()
expected_labels = df['sentiment'].tolist()
n = len(headlines_list)

print(f"Loaded {n} headlines.")
print("Label distribution:")
print(df['sentiment'].value_counts().to_string())
print(f"\nSample headline: {headlines_list[0]}")
print(f"Expected label : {expected_labels[0]}")



In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
HEADLINE_COL = 'headline'
LABEL_COL    = 'sentiment'   # numeric: 0=Positive, 1=Negative, 2=Neutral
RANDOM_STATE = 43            # change for a different random draw
N_SAMPLES    = 80

# ── Numeric → text label mapping ──────────────────────────────────────────────
LABEL_MAP = {0: 'Positive', 1: 'Negative', 2: 'Neutral'}

# ── Random sample ─────────────────────────────────────────────────────────────
sample_df = df.sample(n=N_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)
headlines_list = sample_df[HEADLINE_COL].tolist()

expected_labels = sample_df[LABEL_COL].tolist()

print(f'Randomly sampled {len(headlines_list)} headlines (random_state={RANDOM_STATE}).')
print(f'Expected label distribution:')
import collections
print(dict(collections.Counter(expected_labels)))
print('\nFirst 3 headlines:')
for i, h in enumerate(headlines_list[:3]):
    print(f'  {i+1}. [{expected_labels[i]}] {h[:90]}...')

# ── Save sample to CSV ───────────────────────────────────────────────────────
sample_df.to_csv("/content/random80.csv", index=False, encoding="utf-8")

print(f"Saved sampled dataset to: /content/random80.csv")


## 4 · Label Parser

Extracts the final sentiment word from model output.  
Two-pass: first looks for an explicit `Final sentiment label:` line, then falls back to the last valid label word found anywhere in the response.


In [ ]:
import re

VALID_LABELS = {"Positive", "Negative", "Neutral"}

def parse_reasoning_and_label(text: str):
    if not text:
        return "", "Unknown"

    clean = (
        str(text)
        .replace("\xa0", " ")
        .replace("**", "")
        .replace("*", "")
        .strip()
    )

    lines = [l.strip() for l in clean.splitlines() if l.strip()]
    reasoning = ""
    label = "Unknown"

    # 1) Exact structured lines only
    for line in lines:
        m = re.match(r'(?i)^label\s*:\s*(positive|negative|neutral)\s*$', line)
        if m:
            label = m.group(1).capitalize()

    for line in lines:
        m = re.match(r'(?i)^reasoning\s*:\s*(.+)$', line)
        if m:
            reasoning = m.group(1).strip()

    # 2) Fallback: only search near end of output
    if label == "Unknown":
        tail = clean[-120:]
        matches = re.findall(r'(?i)\b(positive|negative|neutral)\b', tail)
        if matches:
            label = matches[-1].capitalize()

    # 3) Clean reasoning
    reasoning = re.sub(r'(?i)^reasoning\s*:\s*', '', reasoning).strip(" .:-\n\t")
    reasoning = re.sub(r'\s+', ' ', reasoning)

    bad_reasoning = {
        "", "positive", "negative", "neutral",
        "label", "reasoning"
    }
    if reasoning.lower() in bad_reasoning:
        reasoning = ""

    return reasoning, label

## 5 · Run Inference

## **Batch-Processed Financial Sentiment Classification via LLM Inference**

---

### **Quick Breakdown**
* **Prompt Engineering:** Uses a strictly defined **System Prompt** to force the model into a structured "Reasoning + Label" output format.
* **Batching for Speed:** Processes headlines in groups of **8** (`BATCH_SIZE`) to maximize GPU utility and decrease total processing time.
* **Efficient Generation:** Employs `torch.inference_mode()` and `use_cache=True` to strip away unnecessary gradients and speed up token generation.
* **Real-time Monitoring:** Tracks accuracy, processing speed per item, and **ETA** (Estimated Time of Arrival) to keep you updated on the script's progress.
* **Validation:** Decodes generated IDs and parses them into a structured dictionary to compare model predictions against expected labels.

In [ ]:
import time
import torch

# Tokenizer setup
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id



SYSTEM_PROMPT = """
You are a financial news sentiment classifier.

Classify the headline as one label only:
Positive
Negative
Neutral

Output exactly two lines:
Label: Positive, Negative, or Neutral
Reasoning: max 5 words

Rules:
- No extra text
- Do not repeat the headline
- Use only one label
- Never output Unknown
""".strip()

BATCH_SIZE = 8
DEBUG = False
MAX_NEW_TOKENS = 24

device = next(model.parameters()).device
results = []
start = time.time()
n = len(headlines_list)

# Optional: sort by headline length to reduce padding waste
order = sorted(range(n), key=lambda i: len(headlines_list[i]))
sorted_headlines = [headlines_list[i] for i in order]
sorted_expected = [expected_labels[i] for i in order]

for batch_start in range(0, n, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, n)
    batch_headlines = sorted_headlines[batch_start:batch_end]
    batch_expected = sorted_expected[batch_start:batch_end]

    batch_text = [
        f"{SYSTEM_PROMPT}\n\nHeadline: {headline}"
        for headline in batch_headlines
    ]

    inputs = tokenizer(
        batch_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256,
    ).to(device)

    input_lens = inputs["attention_mask"].sum(dim=1).tolist()

    with torch.inference_mode():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            return_dict_in_generate=False,
        )

    batch_results = []
    for j, (headline, exp, input_len) in enumerate(zip(batch_headlines, batch_expected, input_lens)):
        gen_ids = out_ids[j][input_len:]
        response = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

        reasoning, pred = parse_reasoning_and_label(response)
        pred = str(pred).strip().capitalize()
        exp = str(exp).strip().capitalize()
        correct = pred == exp

        original_idx = order[batch_start + j] + 1

        item = {
            "index": original_idx,
            "headline": headline,
            "expected": exp,
            "predicted": pred,
            "correct": correct,
            "reasoning": reasoning,
            "raw_response": response,
        }
        results.append(item)
        batch_results.append(item)

        if DEBUG and pred == "Unknown":
            print(f"\n[{original_idx}/{n}] RAW RESPONSE ERROR: {repr(response)}")

    done = batch_end
    elapsed = time.time() - start
    rate = elapsed / done
    eta = rate * (n - done)
    batch_correct = sum(r["correct"] for r in batch_results)

    print(
        f"[{done:3d}/{n}] "
        f"batch_acc={batch_correct}/{len(batch_results)} "
        f"| {rate:.2f}s/item | ETA≈{eta/60:.1f}min"
    )

# Restore original order for downstream analysis
results = sorted(results, key=lambda x: x["index"])

print(f"\n✓ Done! Total: {(time.time() - start)/60:.1f} min")

In [ ]:
!pip -q install gradio feedparser pandas

5.1 gui

In [ ]:
import time
import re
from urllib.parse import urlparse

import feedparser
import pandas as pd
import gradio as gr
import torch

# -------------------------------------------------------------------
# 1) MODEL / TOKENIZER RUNTIME SETTINGS
# -------------------------------------------------------------------
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

device = next(model.parameters()).device

SYSTEM_PROMPT_TEMPLATE = """
You are an economic-impact sentiment classifier for news headlines.

Classify each headline by its likely broader economic impact.

Use this perspective:
- Positive: improves affordability, savings returns, jobs, investment, growth, supply, stability, or consumer/business outlook
- Negative: worsens affordability, raises costs, reduces supply, creates risk, disruption, losses, legal trouble, or weaker outlook
- Neutral: factual event/update with no clear positive or negative economic direction

Output exactly 2 lines:
Label: Positive, Negative, or Neutral
Reasoning: max 6 words

Rules:
- Consider households, consumers, savers, businesses, markets, property, energy, travel, and the wider economy
- Do not repeat the headline
- Do not output extra text
- Use only one label

Headline: {headline}
""".strip()

# -------------------------------------------------------------------
# 2) FREE RSS FEEDS
# -------------------------------------------------------------------
RSS_FEEDS = {
    "Google News Business (IE)": "https://news.google.com/rss/headlines/section/topic/BUSINESS?hl=en-IE&gl=IE&ceid=IE:en",
    "Google News World (IE)": "https://news.google.com/rss/headlines/section/topic/WORLD?hl=en-IE&gl=IE&ceid=IE:en",
    "Google News Technology (IE)": "https://news.google.com/rss/headlines/section/topic/TECHNOLOGY?hl=en-IE&gl=IE&ceid=IE:en",
    "Google News Top Stories (IE)": "https://news.google.com/rss?hl=en-IE&gl=IE&ceid=IE:en",
    "BBC Top Stories": "http://feeds.bbci.co.uk/news/rss.xml",
    "BBC Business": "http://feeds.bbci.co.uk/news/business/rss.xml",
}

# -------------------------------------------------------------------
# 3) HELPERS
# -------------------------------------------------------------------
NOISE_PREFIXES = [
    "you are an economic-impact sentiment classifier",
    "classify each headline",
    "use this perspective:",
    "output exactly 2 lines:",
    "rules:",
    "headline:",
    "source:",
    "positive:",
    "negative:",
    "neutral:",
]

BAD_REASONING_PREFIXES = [
    "we need to classify",
    "classify the headline",
    "the headline is",
    "headline:",
    "we have to classify",
    "this headline",
    "the sentiment is",
    "sentiment:",
    "label:",
]

TRAILING_SOURCE_PATTERNS = [
    r"\s*-\s*RTE\.ie\s*$",
    r"\s*-\s*The Irish Times\s*$",
    r"\s*-\s*The Irish Independent\s*$",
    r"\s*-\s*BBC News\s*$",
    r"\s*-\s*ESRI\s*$",
]

def shorten_url(url: str, max_len: int = 42):
    if not url:
        return ""
    try:
        p = urlparse(url)
        host = p.netloc.replace("www.", "")
        path = (p.path or "").rstrip("/")
        short = f"{host}{path}" if path else host
        if len(short) > max_len:
            short = short[: max_len - 1] + "…"
        return short
    except Exception:
        return url[: max_len - 1] + "…" if len(url) > max_len else url

def sentiment_chip(label: str):
    label = str(label or "Neutral").capitalize()
    if label == "Positive":
        return "🟢 Positive"
    if label == "Negative":
        return "🔴 Negative"
    return "⚪ Neutral"

def strip_noise_lines(text: str):
    kept = []
    for raw in str(text or "").splitlines():
        line = raw.strip()
        low = line.lower()
        if not line:
            continue
        if any(low.startswith(p) for p in NOISE_PREFIXES):
            continue
        kept.append(line)
    return "\n".join(kept).strip()

def remove_trailing_source(text: str):
    out = str(text or "").strip()
    for pat in TRAILING_SOURCE_PATTERNS:
        out = re.sub(pat, "", out, flags=re.I)
    return out.strip(" .:-\n\t\"'")

def clean_reasoning(reasoning: str, headline: str = ""):
    text = str(reasoning or "").strip()
    if not text:
        return ""

    text = re.sub(r'(?i)^reasoning\s*:\s*', '', text).strip()
    text = re.sub(r'\s+', ' ', text).strip(" .:-\n\t\"'")
    text = remove_trailing_source(text)

    low = text.lower()
    if any(low.startswith(prefix) for prefix in BAD_REASONING_PREFIXES):
        return ""

    if headline:
        h = remove_trailing_source(headline.lower())
        if low == h or low in h:
            return ""

    if len(text) < 4:
        return ""

    bad_fragments = {
        "no financial impact mentioned",
        "mostly factual headline",
        "ongoing news update",
        "12 months to february - the",
        "in trim on the market for",
        "to neobanks - the irish",
        "positive",
        "negative",
        "neutral",
    }
    if text.lower() in bad_fragments:
        return ""

    words = text.split()
    if len(words) > 6:
        text = " ".join(words[:6])

    return text

def parse_reasoning_and_label(text: str, headline: str = ""):
    if not text:
        return "", "Unknown"

    clean = (
        str(text)
        .replace("\xa0", " ")
        .replace("**", "")
        .replace("*", "")
        .strip()
    )
    clean = strip_noise_lines(clean)

    reasoning = ""
    label = "Unknown"
    lines = [l.strip() for l in clean.splitlines() if l.strip()]

    explicit_label_matches = []
    for line in lines:
        m_label = re.match(r'(?i)^label\s*:\s*(positive|negative|neutral)\s*$', line)
        if m_label:
            explicit_label_matches.append(m_label.group(1).capitalize())

        m_reason = re.match(r'(?i)^reasoning\s*:\s*(.+)$', line)
        if m_reason:
            reasoning = m_reason.group(1).strip()

    if explicit_label_matches:
        label = explicit_label_matches[-1]

    if label == "Unknown":
        fallback_patterns = [
            r'(?i)\bprediction\s*:\s*(positive|negative|neutral)\b',
            r'(?i)\bclassified as\s+(positive|negative|neutral)\b',
            r'(?i)\bsentiment\s+is\s+(positive|negative|neutral)\b',
            r'(?i)\bfinal label\s*:\s*(positive|negative|neutral)\b',
        ]
        matches = []
        for pat in fallback_patterns:
            matches.extend(re.findall(pat, clean))
        if matches:
            label = matches[-1].capitalize()

    if label == "Unknown":
        tail = clean[-160:]
        tail_labels = re.findall(r'(?i)\b(positive|negative|neutral)\b', tail)
        if tail_labels:
            label = tail_labels[-1].capitalize()

    reasoning = clean_reasoning(reasoning, headline)
    return reasoning, label

# -------------------------------------------------------------------
# 4) HEADLINE-ONLY ECONOMIC CLASSIFIER
# -------------------------------------------------------------------
def classify_by_economic_impact(headline: str):
    h = remove_trailing_source((headline or "").lower())

    pos = 0
    neg = 0
    neu = 0

    positive_patterns = [
        ("discount", 2),
        ("cost rental", 2),
        ("tenant", 1),
        ("raises rate on savings", 4),
        ("raises savings rate", 4),
        ("higher savings rate", 4),
        ("savings account", 2),
        ("investment", 3),
        ("invests", 3),
        ("expansion", 3),
        ("expand", 2),
        ("job growth", 3),
        ("new jobs", 3),
        ("growth", 2),
        ("raises forecast", 3),
        ("beats estimates", 3),
        ("higher revenue", 3),
        ("support scheme", 2),
        ("affordable", 2),
    ]

    negative_patterns = [
        ("prices rose", 3),
        ("home prices rose", 4),
        ("house prices rose", 4),
        ("rent rises", 4),
        ("warning", 3),
        ("warns", 3),
        ("loss", 3),
        ("losses", 3),
        ("fall", 2),
        ("decline", 2),
        ("cuts", 2),
        ("layoffs", 4),
        ("job cuts", 4),
        ("misses estimates", 4),
        ("fine", 3),
        ("lawsuit", 3),
        ("probe", 3),
        ("downgrade", 3),
        ("crisis", 4),
        ("jet fuel left", 5),
        ("fuel left", 4),
        ("energy watchdog", 1),
        ("disruption", 4),
        ("supply shock", 4),
        ("war", 4),
        ("risk", 2),
    ]

    neutral_patterns = [
        ("on the market", 2),
        ("for sale", 2),
        ("live updates", 2),
        ("latest on", 2),
        ("talks", 1),
        ("statement", 1),
        ("meeting", 1),
        ("report", 1),
        ("appoints", 1),
        ("names", 1),
    ]

    for phrase, weight in positive_patterns:
        if phrase in h:
            pos += weight
    for phrase, weight in negative_patterns:
        if phrase in h:
            neg += weight
    for phrase, weight in neutral_patterns:
        if phrase in h:
            neu += weight

    # phrase-specific overrides
    if "discount" in h and ("tenant" in h or "cost rental" in h):
        pos += 4

    if ("home prices rose" in h or "house prices rose" in h) and "%" in h:
        neg += 3

    if "savings account" in h and "raises rate" in h:
        pos += 4

    if "jet fuel left" in h or ("six weeks" in h and "fuel" in h):
        neg += 5

    if "on the market" in h and ("resort" in h or "hotel" in h or "property" in h):
        neu += 3

    if neg >= pos + 2 and neg >= neu:
        return "Negative"
    if pos >= neg + 2 and pos >= neu:
        return "Positive"
    return "Neutral"

def deterministic_reason(label: str, headline: str):
    h = remove_trailing_source((headline or "").lower())

    if "discount" in h and ("tenant" in h or "cost rental" in h):
        return "lower housing costs help tenants"
    if "savings account" in h and "raises rate" in h:
        return "higher returns for savers"
    if "home prices rose" in h or "house prices rose" in h:
        return "rising housing costs hurt affordability"
    if "jet fuel left" in h or ("six weeks" in h and "fuel" in h):
        return "fuel shortage threatens economic activity"
    if "on the market" in h and ("resort" in h or "property" in h):
        return "asset sale with unclear impact"

    if label == "Positive":
        return "supports broader economic outlook"
    if label == "Negative":
        return "worsens broader economic outlook"
    return "economic direction remains unclear"

# -------------------------------------------------------------------
# 5) RSS FETCH
# -------------------------------------------------------------------
def fetch_rss_headlines(feed_name, max_stories):
    url = RSS_FEEDS[feed_name]
    feed = feedparser.parse(url)

    rows = []
    for entry in feed.entries[:int(max_stories)]:
        title = str(getattr(entry, "title", "") or "").strip()
        link = str(getattr(entry, "link", "") or "").strip()
        published = str(getattr(entry, "published", "") or "").strip()

        source_name = feed_name
        source_obj = getattr(entry, "source", None)
        if source_obj:
            try:
                source_name = source_obj.get("title", feed_name) or feed_name
            except Exception:
                source_name = feed_name

        if title:
            rows.append({
                "source": source_name,
                "headline": title,
                "published": published,
                "url": link,
                "short_url": shorten_url(link),
            })

    return rows

# -------------------------------------------------------------------
# 6) BATCH INFERENCE
# -------------------------------------------------------------------
def analyze_headlines_batch(headlines, sources=None, batch_size=8, max_new_tokens=16):
    sources = sources or [""] * len(headlines)

    order = sorted(range(len(headlines)), key=lambda i: len(headlines[i]))
    sorted_headlines = [headlines[i] for i in order]
    sorted_sources = [sources[i] for i in order]

    collected = [None] * len(headlines)

    for batch_start in range(0, len(sorted_headlines), batch_size):
        batch_end = min(batch_start + batch_size, len(sorted_headlines))
        batch_headlines = sorted_headlines[batch_start:batch_end]
        batch_sources = sorted_sources[batch_start:batch_end]

        batch_prompts = []
        for h, s in zip(batch_headlines, batch_sources):
            short_h = h[:180]
            prompt = SYSTEM_PROMPT_TEMPLATE.format(headline=short_h)
            if s:
                prompt = f"Source: {s}\n{prompt}"
            batch_prompts.append(prompt)

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256,
        ).to(device)

        input_lens = inputs["attention_mask"].sum(dim=1).tolist()

        with torch.inference_mode():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                min_new_tokens=3,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                return_dict_in_generate=False,
            )

        for j, input_len in enumerate(input_lens):
            headline = batch_headlines[j]
            gen_ids = out_ids[j][input_len:]
            response = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

            parsed_reasoning, parsed_label = parse_reasoning_and_label(response, headline)
            rule_label = classify_by_economic_impact(headline)

            if parsed_label in {"Positive", "Negative", "Neutral"}:
                final_label = parsed_label
            else:
                final_label = rule_label

            if not parsed_reasoning:
                final_reasoning = deterministic_reason(final_label, headline)
            else:
                final_reasoning = parsed_reasoning

            bad_reason_outputs = {
                "no financial impact mentioned",
                "mostly factual headline",
                "ongoing news update",
                "economic direction remains unclear",
                "12 months to february - the",
                "in trim on the market for",
            }
            if final_reasoning.lower() in bad_reason_outputs:
                final_reasoning = deterministic_reason(final_label, headline)

            # Override obviously weak model outputs with stronger rule-based economic lens
            if rule_label != parsed_label:
                if "discount" in headline.lower() or "savings account" in headline.lower() \
                   or "home prices rose" in headline.lower() or "jet fuel" in headline.lower():
                    final_label = rule_label
                    final_reasoning = deterministic_reason(final_label, headline)

            collected[order[batch_start + j]] = {
                "predicted": final_label,
                "reasoning": final_reasoning,
                "raw_response": response,
            }

    return collected

# -------------------------------------------------------------------
# 7) MAIN APP FUNCTION
# -------------------------------------------------------------------
def fetch_and_analyze(feed_name, max_stories):
    start = time.time()

    try:
        stories = fetch_rss_headlines(feed_name, max_stories)

        if not stories:
            empty_df = pd.DataFrame([{
                "source": feed_name,
                "headline": "No stories found.",
                "published": "",
                "predicted": "Neutral",
                "reasoning": "no rss items returned",
                "url": "",
                "short_url": "",
                "sentiment": "⚪ Neutral",
            }])
            status = "No stories returned from RSS feed."
            return status, empty_df[["source", "headline", "sentiment", "reasoning", "published", "short_url"]]

        headlines = [s["headline"] for s in stories]
        sources = [s["source"] for s in stories]

        preds = analyze_headlines_batch(
            headlines=headlines,
            sources=sources,
            batch_size=8,
            max_new_tokens=16,
        )

        rows = []
        counts = {"Positive": 0, "Negative": 0, "Neutral": 0}

        for story, pred in zip(stories, preds):
            label = pred["predicted"]
            counts[label] = counts.get(label, 0) + 1

            rows.append({
                "source": story["source"],
                "headline": story["headline"],
                "published": story["published"],
                "predicted": label,
                "sentiment": sentiment_chip(label),
                "reasoning": pred["reasoning"],
                "url": story["url"],
                "short_url": story["short_url"],
            })

        df = pd.DataFrame(rows)
        elapsed = time.time() - start

        status = (
            f"**{len(df)} stories analyzed** in {elapsed:.1f}s  \n"
            f"🟢 {counts.get('Positive', 0)} positive · "
            f"🔴 {counts.get('Negative', 0)} negative · "
            f"⚪ {counts.get('Neutral', 0)} neutral"
        )

        return status, df[["source", "headline", "sentiment", "reasoning", "published", "short_url"]]

    except Exception as e:
        err_df = pd.DataFrame([{
            "source": "ERROR",
            "headline": str(e),
            "published": "",
            "sentiment": "⚪ Neutral",
            "reasoning": "callback failed",
            "short_url": "",
        }])
        return f"**Error:** {e}", err_df[["source", "headline", "sentiment", "reasoning", "published", "short_url"]]

# -------------------------------------------------------------------
# 8) SIMPLE MODERN UI
# -------------------------------------------------------------------
CUSTOM_CSS = """
.gradio-container {
    max-width: 1180px !important;
    margin: 0 auto;
    background: #f8f8f7;
}

.block-title {
    text-align: center;
    font-weight: 700;
    font-size: 1.05rem;
    color: #111827;
    margin-bottom: 10px;
}

.block-note {
    text-align: center;
    color: #6b7280;
    font-size: 0.93rem;
    line-height: 1.5;
    margin-bottom: 10px;
}

.simple-card {
    border: 1px solid #e5e7eb;
    border-radius: 16px;
    background: white;
    padding: 10px;
    box-shadow: 0 1px 2px rgba(0,0,0,0.03);
}

.app-header {
    text-align: center;
    padding: 8px 0 16px 0;
}

.app-title {
    font-size: 2rem;
    font-weight: 700;
    color: #111827;
    letter-spacing: -0.03em;
    margin-bottom: 6px;
}

.app-subtitle {
    color: #6b7280;
    font-size: 0.98rem;
    line-height: 1.6;
    max-width: 760px;
    margin: 0 auto;
}

.gr-button-primary {
    border-radius: 12px !important;
    min-height: 46px !important;
    font-weight: 600 !important;
}

.status-text {
    text-align: center;
    font-size: 0.98rem;
    line-height: 1.6;
}
"""

with gr.Blocks(theme=gr.themes.Soft(), css=CUSTOM_CSS) as demo:
    gr.HTML("""
    <div class="app-header">
        <div class="app-title">Economic Impact Monitor</div>
        <div class="app-subtitle">
            Live RSS headlines classified with a broader economic lens.
        </div>
    </div>
    """)

    with gr.Row(equal_height=False):
        with gr.Column(scale=1, min_width=320):
            with gr.Column(elem_classes="simple-card"):
                gr.HTML('<div class="block-title">Controls</div>')
                gr.HTML('<div class="block-note">Choose a source and number of stories.</div>')

                feed_name = gr.Dropdown(
                    choices=list(RSS_FEEDS.keys()),
                    value="Google News Business (IE)",
                    label="Source",
                    interactive=True
                )

                max_stories = gr.Slider(
                    minimum=1,
                    maximum=12,
                    step=1,
                    value=6,
                    label="Stories",
                    interactive=True
                )

                run_btn = gr.Button("Analyze headlines", variant="primary")

            with gr.Column(elem_classes="simple-card"):
                gr.HTML('<div class="block-title">Status</div>')
                status_box = gr.Markdown('<div class="status-text">Ready to analyze.</div>')

        with gr.Column(scale=2, min_width=760):
            with gr.Column(elem_classes="simple-card"):
                gr.HTML('<div class="block-title">Results</div>')
                gr.HTML('<div class="block-note">Source → Headline → Sentiment → Reasoning → Date → Link</div>')

                output_table = gr.Dataframe(
                    headers=["source", "headline", "sentiment", "reasoning", "published", "short_url"],
                    datatype=["str", "str", "str", "str", "str", "str"],
                    label="",
                    wrap=True,
                    interactive=False,
                )

    run_btn.click(
        fn=fetch_and_analyze,
        inputs=[feed_name, max_stories],
        outputs=[status_box, output_table],
    )

demo.queue()
demo.launch(share=True, debug=True)

## 6 · Accuracy & Confusion Matrix

In [ ]:

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

results_df = pd.DataFrame(results)
y_true = results_df['expected'].tolist()
y_pred = results_df['predicted'].tolist()

acc       = accuracy_score(y_true, y_pred)
n_correct = results_df['correct'].sum()
n_unknown = (results_df['predicted'] == 'Unknown').sum()

print("=" * 60)
print(f"  Overall Accuracy : {acc:.1%}  ({n_correct}/{len(results_df)} correct)")
print(f"  Unparsed labels  : {n_unknown}")
print("=" * 60)
print()
print(classification_report(
    y_true, y_pred,
    labels=['Positive', 'Negative', 'Neutral'],
    zero_division=0
))

# ── Confusion matrix ──────────────────────────────────────────────────────────
labels = ['Positive', 'Negative', 'Neutral']
cm = confusion_matrix(y_true, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, linewidths=0.5, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Expected',  fontsize=12)
ax.set_title(f'Confusion Matrix  —  Accuracy: {acc:.1%}', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print("Saved → confusion_matrix.png")


## 7 · Misclassified Headlines

In [ ]:
wrong_df = results_df[results_df['correct'] == False][
    ['index', 'headline', 'expected', 'predicted', 'reasoning']
].reset_index(drop=True)

print(f"Misclassified: {len(wrong_df)} / {len(results_df)}")
print()
for _, row in wrong_df.iterrows():
    print(f"#{int(row['index']):3d} | Expected={row['expected']:<9s} Predicted={row['predicted']}")
    print(f"       {row['headline'][:110]}")
    print(f"       Reasoning: {row['reasoning'][:120]}")
    print()


## 8 · Save Results

In [ ]:
out = '/content/sentiment_results.csv'
results_df[['index', 'headline', 'expected', 'predicted', 'correct']].to_csv(out, index=False)
print(f"Results saved → {out}")

print("\nLabel distribution comparison:")
comp = pd.DataFrame({
    'Expected': results_df['expected'].value_counts(),
    'Predicted': results_df['predicted'].value_counts(),
}).fillna(0).astype(int)

print(comp)
#print(comp.to_string())
